# T20 World Cup Bowling Analysis

## Comprehensive analysis of bowling performances across T20 World Cups (2014-2024)

This notebook explores:
- Top wicket takers and economy rates
- Bowling consistency and effectiveness
- Powerplay vs death bowling specialists
- Wicket-taking patterns
- Bowler comparisons and rankings

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries loaded")

In [ ]:
# Load processed data
bowling_stats = pd.read_csv('../data/processed/player_bowling_stats.csv')
all_deliveries = pd.read_csv('../data/processed/all_deliveries.csv')
match_summaries = pd.read_csv('../data/processed/match_summaries.csv')

print(f"📊 Loaded data:")
print(f"  - Bowling stats: {len(bowling_stats)} bowlers")
print(f"  - Deliveries: {len(all_deliveries):,} balls")
print(f"  - Matches: {len(match_summaries)} matches")

## 1. Top Wicket Takers Analysis

In [ ]:
# Filter qualified bowlers (minimum 10 wickets)
qualified = bowling_stats[bowling_stats['wickets'] >= 10].copy()

print("🏆 Top 20 Wicket Takers (min 10 wickets)")
print("=" * 80)
top_bowlers = qualified.nlargest(20, 'wickets')
print(top_bowlers[['player', 'wickets', 'matches', 'overs', 'economy', 'average', 'strike_rate']].to_string(index=False))

In [ ]:
# Visualization: Top 15 Wicket Takers
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Total wickets
top_15 = bowling_stats.nlargest(15, 'wickets')
colors = ['darkred' if e < 7 else 'steelblue' for e in top_15['economy']]
ax1.barh(range(len(top_15)), top_15['wickets'], color=colors)
ax1.set_yticks(range(len(top_15)))
ax1.set_yticklabels(top_15['player'])
ax1.set_xlabel('Total Wickets', fontsize=12)
ax1.set_title('Top 15 Wicket Takers - T20 World Cups (2014-2024)', fontsize=14, fontweight='bold')
ax1.invert_yaxis()

# Add value labels with economy
for i, (wickets, econ) in enumerate(zip(top_15['wickets'], top_15['economy'])):
    ax1.text(wickets + 0.5, i, f"{wickets} ({econ:.2f})", va='center', fontsize=9)

# Economy vs Average scatter
scatter = ax2.scatter(qualified['economy'], qualified['average'], 
                     s=qualified['wickets']*5, alpha=0.6, c=qualified['wickets'], 
                     cmap='coolwarm_r')
ax2.set_xlabel('Economy Rate', fontsize=12)
ax2.set_ylabel('Bowling Average', fontsize=12)
ax2.set_title('Economy vs Average (min 10 wickets)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

# Annotate top performers
for _, row in qualified.nlargest(5, 'wickets').iterrows():
    ax2.annotate(row['player'], (row['economy'], row['average']), 
                fontsize=8, alpha=0.7)

plt.colorbar(scatter, ax=ax2, label='Wickets')
plt.tight_layout()
plt.show()

## 2. Economy Rate Analysis

In [ ]:
# Best economy rates (minimum 10 wickets)
print("💰 Best Economy Rates (min 10 wickets)")
print("=" * 80)
best_economy = qualified.nsmallest(15, 'economy')
print(best_economy[['player', 'wickets', 'balls_bowled', 'runs_conceded', 'economy', 'average']].to_string(index=False))

In [ ]:
# Visualization: Economy distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Economy rate distribution
ax1.hist(qualified['economy'], bins=25, color='steelblue', edgecolor='black', alpha=0.7)
ax1.axvline(qualified['economy'].median(), color='red', linestyle='--', 
           label=f"Median: {qualified['economy'].median():.2f}")
ax1.axvline(qualified['economy'].mean(), color='orange', linestyle='--', 
           label=f"Mean: {qualified['economy'].mean():.2f}")
ax1.set_xlabel('Economy Rate', fontsize=12)
ax1.set_ylabel('Number of Bowlers', fontsize=12)
ax1.set_title('Distribution of Economy Rates (min 10 wickets)', fontsize=14, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Best economy bowlers
best_econ = qualified.nsmallest(12, 'economy')
ax2.barh(range(len(best_econ)), best_econ['economy'], color='darkgreen')
ax2.set_yticks(range(len(best_econ)))
ax2.set_yticklabels(best_econ['player'])
ax2.set_xlabel('Economy Rate', fontsize=12)
ax2.set_title('Most Economical Bowlers', fontsize=14, fontweight='bold')
ax2.invert_yaxis()

# Add value labels
for i, (econ, wickets) in enumerate(zip(best_econ['economy'], best_econ['wickets'])):
    ax2.text(econ + 0.1, i, f"{econ:.2f} ({int(wickets)}W)", va='center', fontsize=9)

plt.tight_layout()
plt.show()

## 3. Wicket-Taking Efficiency

In [ ]:
# Calculate efficiency metrics
qualified['wickets_per_match'] = (qualified['wickets'] / qualified['matches']).round(2)
qualified['balls_per_wicket'] = qualified['strike_rate']

print("⚡ Most Effective Wicket Takers (min 10 wickets, sorted by strike rate)")
print("=" * 80)
effective = qualified.nsmallest(15, 'strike_rate')
print(effective[['player', 'wickets', 'matches', 'wickets_per_match', 'strike_rate', 'economy', 'average']].to_string(index=False))

In [ ]:
# Categorize bowlers
qualified['category'] = 'Average'
qualified.loc[(qualified['economy'] < 7) & (qualified['wickets'] >= 20), 'category'] = 'Elite'
qualified.loc[(qualified['economy'] < 7.5) | (qualified['wickets'] >= 15), 'category'] = 'Good'

print("\n📊 Bowler Categories:")
print(qualified['category'].value_counts())
print("\n🌟 Elite Bowlers (Economy < 7 & Wickets >= 20):")
elite = qualified[qualified['category'] == 'Elite'].sort_values('wickets', ascending=False)
print(elite[['player', 'wickets', 'economy', 'average', 'strike_rate']].to_string(index=False))

## 4. Dismissal Types Analysis

In [ ]:
# Analyze types of dismissals
dismissals = all_deliveries[all_deliveries['dismissal'].notna()]

print("🎯 Types of Dismissals")
print("=" * 60)
dismissal_counts = dismissals['dismissal'].value_counts()
dismissal_pct = (dismissal_counts / dismissal_counts.sum() * 100).round(1)

for dismissal_type, count in dismissal_counts.items():
    pct = dismissal_pct[dismissal_type]
    print(f"  {dismissal_type:20s}: {count:4d} ({pct:5.1f}%)")

print(f"\nTotal Wickets: {len(dismissals)}")

In [ ]:
# Visualization: Dismissal types pie chart
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Pie chart of dismissal types
top_dismissals = dismissal_counts.head(8)
colors = plt.cm.Set3(range(len(top_dismissals)))
ax1.pie(top_dismissals, labels=top_dismissals.index, autopct='%1.1f%%', 
       colors=colors, startangle=90)
ax1.set_title('Distribution of Dismissal Types', fontsize=14, fontweight='bold')

# Wickets per over distribution
dismissals_copy = dismissals.copy()
dismissals_copy['over'] = dismissals_copy['ball'].apply(lambda x: int(x) if pd.notna(x) else 0)
wickets_per_over = dismissals_copy['over'].value_counts().sort_index()

ax2.bar(wickets_per_over.index, wickets_per_over.values, color='steelblue', edgecolor='black')
ax2.set_xlabel('Over Number', fontsize=12)
ax2.set_ylabel('Number of Wickets', fontsize=12)
ax2.set_title('Wickets Distribution by Over', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 5. Key Insights & Summary

In [ ]:
print("="*80)
print("⚾ KEY BOWLING INSIGHTS - T20 WORLD CUPS (2014-2024)")
print("="*80)

# Overall stats
total_wickets = all_deliveries['dismissal'].notna().sum()
print(f"\n📈 Overall Statistics:")
print(f"  Total wickets: {total_wickets}")
print(f"  Average economy rate: {qualified['economy'].mean():.2f}")
print(f"  Average strike rate: {qualified['strike_rate'].mean():.2f} balls per wicket")
print(f"  Wickets per match: {total_wickets / len(match_summaries):.1f}")

# Top wicket taker
top_bowler = bowling_stats.iloc[0]
print(f"\n👑 Leading Wicket Taker:")
print(f"  {top_bowler['player']}: {int(top_bowler['wickets'])} wickets in {int(top_bowler['matches'])} matches")
print(f"  Economy: {top_bowler['economy']:.2f} | Average: {top_bowler['average']:.2f}")

# Best economy
best_econ = qualified.nsmallest(1, 'economy').iloc[0]
print(f"\n💰 Best Economy Rate (min 10 wickets):")
print(f"  {best_econ['player']}: {best_econ['economy']:.2f} ({int(best_econ['wickets'])} wickets)")

# Best strike rate
best_sr = qualified.nsmallest(1, 'strike_rate').iloc[0]
print(f"\n⚡ Best Strike Rate (min 10 wickets):")
print(f"  {best_sr['player']}: {best_sr['strike_rate']:.2f} balls per wicket")

# Most common dismissal
print(f"\n🎯 Most Common Dismissal:")
print(f"  {dismissal_counts.index[0]}: {dismissal_counts.iloc[0]} ({dismissal_pct.iloc[0]:.1f}%)")

print("\n" + "="*80)

## Next Steps

- Analyze powerplay vs death bowling specialists
- Compare spinners vs pacers performance
- Venue-specific bowling analysis
- Match situation bowling (defending vs chasing)